# Blood Atlas Data Preparation

The full HCA contributor-generated PBMC AnnData file is not included in this repository due to size.

External source files:
- all_pbmcs_rna.h5ad
- all_pbmcs_metadata.csv

The benchmark input `blood_atlas_benchmark.h5ad` is generated here. This notebook is run prior to the benchmark workflow. Users do NOT need to run this notebook, as we provide the processed adata in the Zenodo snapshot.

Processing steps:
1. Load expression matrix from `all_pbmcs_rna.h5ad`.
2. Load metadata from `all_pbmcs_metadata.csv`.
3. Attach metadata by row order.
4. Standardize columns:
   - `Donor_id` → `donor_id`
   - `Cluster_names` → `cell_type`
   - `Batch` → `batch`
   - `File_name` → `sample_id`
5. Remove missing donor or cell-type labels.
6. Apply fixed donor/cell-type-aware subsampling with seed 0.
7. Save benchmark-ready AnnData as `blood_atlas_subsampled.h5ad`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc

In [ ]:
# Paths
repo_root = Path("/users/xchen5/scRNA-cross-donor-generalization")

raw_dir = Path("/fastscratch/myscratch/xchen5/all_pbmcs")
adata_path = raw_dir / "all_pbmcs_rna.h5ad"
meta_path = raw_dir / "all_pbmcs_metadata.csv"

out_dir = repo_root / "data" / "blood_atlas"
out_dir.mkdir(parents=True, exist_ok=True)

out_h5ad = out_dir / "blood_atlas_subsampled.h5ad"

In [3]:
# Load full expression matrix and metadata
adata = sc.read_h5ad(adata_path)
meta = pd.read_csv(meta_path)

print(adata)
print(meta.shape)
print(meta.columns.tolist())

AnnData object with n_obs × n_vars = 1916367 × 36601
(1916367, 20)
['Unnamed: 0', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nCount_HTO', 'nFeature_HTO', 'percent.mt', 'percent.ribo', 'log2_nCount', 'log2_nFeature', 'log2_mt', 'Donor_id', 'Age_group', 'Sex', 'Age', 'Tube_id', 'Batch', 'File_name', 'Cluster_names', 'Cluster_numbers']


In [4]:
# Basic sanity check
assert adata.n_obs == meta.shape[0], "Metadata rows do not match AnnData cells"

# Attach metadata by row order
meta = meta.copy()
meta.index = adata.obs_names
adata.obs = meta

print(adata)
print(adata.obs.head())

AnnData object with n_obs × n_vars = 1916367 × 36601
    obs: 'Unnamed: 0', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nCount_HTO', 'nFeature_HTO', 'percent.mt', 'percent.ribo', 'log2_nCount', 'log2_nFeature', 'log2_mt', 'Donor_id', 'Age_group', 'Sex', 'Age', 'Tube_id', 'Batch', 'File_name', 'Cluster_names', 'Cluster_numbers'
                                                      Unnamed: 0  \
ALAW-AS044-1_AAAGCAAGTAGCTTGT-1  ALAW-AS044-1_AAACCTGAGAAGCCCA-1   
ALAW-AS044-1_AAAGCAATCTCAAGTG-1  ALAW-AS044-1_AAACCTGAGAGTAATC-1   
ALAW-AS044-1_AAAGTAGGTGTTGAGG-1  ALAW-AS044-1_AAACCTGAGCAATCTC-1   
ALAW-AS044-1_AACCATGCACCGATAT-1  ALAW-AS044-1_AAACCTGAGGTACTCT-1   
ALAW-AS044-1_AACCATGCATGCCACG-1  ALAW-AS044-1_AAACCTGCAATGTAAG-1   

                                    orig.ident  nCount_RNA  nFeature_RNA  \
ALAW-AS044-1_AAAGCAAGTAGCTTGT-1  SeuratProject        4145          1745   
ALAW-AS044-1_AAAGCAATCTCAAGTG-1  SeuratProject        1921           960   
ALAW-AS044-1_AAAGTAGGTGTTGAGG-1  

In [5]:
# Standardize benchmark metadata columns
adata.obs["donor_id"] = adata.obs["Donor_id"].astype(str)
adata.obs["cell_type"] = adata.obs["Cluster_names"].astype(str)
adata.obs["batch"] = adata.obs["Batch"].astype(str)
adata.obs["sample_id"] = adata.obs["File_name"].astype(str)

adata.obs["age"] = pd.to_numeric(adata.obs["Age"], errors="coerce")
adata.obs["age_group"] = adata.obs["Age_group"].astype(str)
adata.obs["sex"] = adata.obs["Sex"].astype(str)

In [6]:
# Remove cells with missing essential labels
required = ["donor_id", "cell_type", "batch"]

mask = np.ones(adata.n_obs, dtype=bool)
for col in required:
    vals = adata.obs[col].astype(str)
    mask &= vals.notna()
    mask &= ~vals.isin(["nan", "None", "NA", ""])

adata = adata[mask].copy()

print(adata)
print("n donors:", adata.obs["donor_id"].nunique())
print("n cell types:", adata.obs["cell_type"].nunique())
print("n batches:", adata.obs["batch"].nunique())

AnnData object with n_obs × n_vars = 1916367 × 36601
    obs: 'Unnamed: 0', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nCount_HTO', 'nFeature_HTO', 'percent.mt', 'percent.ribo', 'log2_nCount', 'log2_nFeature', 'log2_mt', 'Donor_id', 'Age_group', 'Sex', 'Age', 'Tube_id', 'Batch', 'File_name', 'Cluster_names', 'Cluster_numbers', 'donor_id', 'cell_type', 'batch', 'sample_id', 'age', 'age_group', 'sex'
n donors: 166
n cell types: 9
n batches: 14


In [7]:
# Summarize full dataset before filtering/subsampling
full_celltype_counts = adata.obs["cell_type"].value_counts()
full_donor_counts = adata.obs["donor_id"].value_counts()
full_batch_counts = adata.obs["batch"].value_counts()

display(full_celltype_counts)
display(full_donor_counts.head())
display(full_batch_counts)

cell_type
CD4+ T cells             901152
Myeloid cells            336935
TRAV1-2- CD8+ T cells    313343
NK cells                 205469
B cells                   71614
gd T cells                60325
MAIT cells                24245
Progenitor cells           1794
DN T cells                 1490
Name: count, dtype: int64

donor_id
E23    26043
E21    23049
C13    22604
D05    22589
A23    22245
Name: count, dtype: int64

batch
AS059    163511
AS056    162103
AS053    159286
AS062    155431
AS058    154747
AS060    153517
AS055    150823
AS061    146351
AS054    142318
AS052    131541
AS057    129960
AS049    121376
AS051    119816
AS044     25587
Name: count, dtype: int64

In [8]:
min_cells = 5000
min_donors = 20
max_cells_per_group = 100
seed = 0

celltype_summary = (
    adata.obs
    .groupby("cell_type", observed=True)
    .agg(
        n_cells=("cell_type", "size"),
        n_donors=("donor_id", "nunique"),
    )
    .sort_values(["n_donors", "n_cells"], ascending=False)
)

keep_celltypes = celltype_summary.index[
    (celltype_summary["n_cells"] >= min_cells) &
    (celltype_summary["n_donors"] >= min_donors)
].tolist()

display(celltype_summary)
print("Keeping:", keep_celltypes)

adata_filt = adata[adata.obs["cell_type"].isin(keep_celltypes)].copy()

,n_cells,n_donors
cell_type,,
CD4+ T cells,901152,166
Myeloid cells,336935,166
TRAV1-2- CD8+ T cells,313343,166
NK cells,205469,166
B cells,71614,166
gd T cells,60325,166
MAIT cells,24245,166
DN T cells,1490,163
Progenitor cells,1794,160


Keeping: ['CD4+ T cells', 'Myeloid cells', 'TRAV1-2- CD8+ T cells', 'NK cells', 'B cells', 'gd T cells', 'MAIT cells']


In [9]:
import numpy as np

def subsample_by_group(adata, group_cols, max_cells_per_group=100, seed=0):
    rng = np.random.default_rng(seed)
    keep = []

    groups = adata.obs.groupby(group_cols, observed=True).indices

    for _, idx in groups.items():
        idx = np.asarray(idx)
        if len(idx) > max_cells_per_group:
            idx = rng.choice(idx, size=max_cells_per_group, replace=False)
        keep.extend(idx)

    keep = np.asarray(keep)
    rng.shuffle(keep)

    return adata[keep].copy()

adata_bench = subsample_by_group(
    adata_filt,
    group_cols=["donor_id", "cell_type"],
    max_cells_per_group=max_cells_per_group,
    seed=seed,
)

print(adata_bench)
print(adata_bench.obs["cell_type"].value_counts())
print("n donors:", adata_bench.obs["donor_id"].nunique())
print("n batches:", adata_bench.obs["batch"].nunique())

AnnData object with n_obs × n_vars = 108682 × 36601
    obs: 'Unnamed: 0', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nCount_HTO', 'nFeature_HTO', 'percent.mt', 'percent.ribo', 'log2_nCount', 'log2_nFeature', 'log2_mt', 'Donor_id', 'Age_group', 'Sex', 'Age', 'Tube_id', 'Batch', 'File_name', 'Cluster_names', 'Cluster_numbers', 'donor_id', 'cell_type', 'batch', 'sample_id', 'age', 'age_group', 'sex'
cell_type
Myeloid cells            16600
CD4+ T cells             16600
TRAV1-2- CD8+ T cells    16575
NK cells                 16464
B cells                  16297
gd T cells               14085
MAIT cells               12061
Name: count, dtype: int64
n donors: 166
n batches: 14


In [10]:
# Final subset sanity checks
print(adata_bench)

print("\nCell type counts:")
display(adata_bench.obs["cell_type"].value_counts())

print("\nDonor counts summary:")
display(adata_bench.obs["donor_id"].value_counts().describe())

print("\nBatch counts:")
display(adata_bench.obs["batch"].value_counts())

print("\nAge summary:")
display(adata_bench.obs[["donor_id", "age", "age_group", "sex"]].drop_duplicates().describe(include="all"))

AnnData object with n_obs × n_vars = 108682 × 36601
    obs: 'Unnamed: 0', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nCount_HTO', 'nFeature_HTO', 'percent.mt', 'percent.ribo', 'log2_nCount', 'log2_nFeature', 'log2_mt', 'Donor_id', 'Age_group', 'Sex', 'Age', 'Tube_id', 'Batch', 'File_name', 'Cluster_names', 'Cluster_numbers', 'donor_id', 'cell_type', 'batch', 'sample_id', 'age', 'age_group', 'sex'

Cell type counts:


cell_type
Myeloid cells            16600
CD4+ T cells             16600
TRAV1-2- CD8+ T cells    16575
NK cells                 16464
B cells                  16297
gd T cells               14085
MAIT cells               12061
Name: count, dtype: int64


Donor counts summary:


count    166.000000
mean     654.710843
std       54.067529
min      400.000000
25%      629.250000
50%      669.000000
75%      700.000000
max      700.000000
Name: count, dtype: float64


Batch counts:


batch
AS061    10806
AS059     9935
AS055     9005
AS056     8802
AS058     8497
AS060     8293
AS054     7874
AS051     7685
AS049     7511
AS053     7410
AS062     7341
AS057     7256
AS052     7107
AS044     1160
Name: count, dtype: int64


Age summary:


,donor_id,age,age_group,sex
count,313,313.000000,313,313
unique,166,NaN,5,2
top,E04,NaN,A,Male
freq,3,NaN,95,266
mean,NaN,49.472843,NaN,NaN
std,NaN,16.916463,NaN,NaN
min,NaN,25.000000,NaN,NaN
25%,NaN,33.000000,NaN,NaN
50%,NaN,50.000000,NaN,NaN
75%,NaN,63.000000,NaN,NaN


In [11]:
# Check donor-batch structure
donor_batch = (
    adata_bench.obs[["donor_id", "batch"]]
    .drop_duplicates()
    .groupby("donor_id")["batch"]
    .nunique()
)

print("Number of batches per donor:")
display(donor_batch.value_counts().sort_index())

multi_batch_donors = donor_batch[donor_batch > 1]
print("Donors appearing in multiple batches:", len(multi_batch_donors))
display(multi_batch_donors.head())

Number of batches per donor:


batch
1    67
2    58
3    41
Name: count, dtype: int64

Donors appearing in multiple batches: 99


donor_id
A01    2
A02    2
A03    3
A04    3
A05    2
Name: batch, dtype: int64

In [12]:
# Donor coverage per cell type after subsampling
celltype_summary_final = (
    adata_bench.obs
    .groupby("cell_type", observed=True)
    .agg(
        n_cells=("cell_type", "size"),
        n_donors=("donor_id", "nunique"),
        n_batches=("batch", "nunique"),
    )
    .sort_values("n_cells", ascending=False)
)

display(celltype_summary_final)

,n_cells,n_donors,n_batches
cell_type,,,
CD4+ T cells,16600,166,14
Myeloid cells,16600,166,14
TRAV1-2- CD8+ T cells,16575,166,14
NK cells,16464,166,14
B cells,16297,166,14
gd T cells,14085,166,14
MAIT cells,12061,166,14


In [13]:
# Reduce file size
categorical_cols = [
    "donor_id",
    "cell_type",
    "batch",
    "sample_id",
    "age_group",
    "sex",
    "Donor_id",
    "Age_group",
    "Sex",
    "Tube_id",
    "Batch",
    "File_name",
    "Cluster_names",
]

for col in categorical_cols:
    if col in adata_bench.obs.columns:
        adata_bench.obs[col] = adata_bench.obs[col].astype("category")

In [14]:
# Save summary tables
adata_bench.obs["cell_type"].value_counts().to_csv(
    out_dir / "celltype_counts.csv",
    header=["n_cells"],
)

adata_bench.obs["donor_id"].value_counts().to_csv(
    out_dir / "donor_counts.csv",
    header=["n_cells"],
)

adata_bench.obs["batch"].value_counts().to_csv(
    out_dir / "batch_counts.csv",
    header=["n_cells"],
)

celltype_summary_final.to_csv(out_dir / "celltype_summary.csv")

donor_summary = (
    adata_bench.obs
    .groupby("donor_id", observed=True)
    .agg(
        n_cells=("donor_id", "size"),
        n_cell_types=("cell_type", "nunique"),
        batch=("batch", lambda x: ",".join(sorted(map(str, x.unique())))),
        age=("age", "first"),
        age_group=("age_group", "first"),
        sex=("sex", "first"),
    )
    .sort_values("n_cells", ascending=False)
)

donor_summary.to_csv(out_dir / "donor_summary.csv")
display(donor_summary.head())

,n_cells,n_cell_types,batch,age,age_group,sex
donor_id,,,,,,
A09,700,7,"AS053,AS054,AS057",35,A,Male
FE05,700,7,AS056,73,E,Female
FC01,700,7,AS054,52,C,Female
FC02,700,7,"AS054,AS055",50,C,Female
FC04,700,7,AS055,51,C,Female


In [17]:
# Save AnnData
adata_bench.write_h5ad(out_h5ad, compression="gzip")

print("Saved:", out_h5ad)

Saved: /users/xchen5/scRNA-cross-donor-generalization/data/blood_atlas/blood_atlas_subsampled.h5ad


In [18]:
prep_summary = pd.DataFrame({
    "parameter": [
        "source_adata_path",
        "source_metadata_path",
        "n_cells_original",
        "n_genes",
        "min_cells_per_celltype",
        "min_donors_per_celltype",
        "max_cells_per_donor_celltype",
        "subsample_seed",
        "n_cells_final",
        "n_donors_final",
        "n_cell_types_final",
        "n_batches_final",
        "output_h5ad",
    ],
    "value": [
        str(adata_path),
        str(meta_path),
        meta.shape[0],
        adata_bench.n_vars,
        min_cells,
        min_donors,
        max_cells_per_group,
        seed,
        adata_bench.n_obs,
        adata_bench.obs["donor_id"].nunique(),
        adata_bench.obs["cell_type"].nunique(),
        adata_bench.obs["batch"].nunique(),
        str(out_h5ad),
    ],
})

prep_summary.to_csv(out_dir / "preprocessing_summary.csv", index=False)
display(prep_summary)

,parameter,value
0,source_adata_path,/fastscratch/myscratch/xchen5/all_pbmcs/all_pb...
1,source_metadata_path,/fastscratch/myscratch/xchen5/all_pbmcs/all_pb...
2,n_cells_original,1916367
3,n_genes,36601
4,min_cells_per_celltype,5000
5,min_donors_per_celltype,20
6,max_cells_per_donor_celltype,100
7,subsample_seed,0
8,n_cells_final,108682
9,n_donors_final,166
